In [5]:
# =========================
# RAG SYSTEM (OPENROUTER)
# =========================

# Install if needed:
# pip install pandas scikit-learn openai

import pandas as pd
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 1. CONFIG
# =========================

API_KEY = "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"  # <-- PUT YOUR KEY HERE
MODEL_NAME = "qwen/qwen3.6-plus:free"

# =========================
# 2. LOAD DATA
# =========================

df = pd.read_csv("drug_docs.csv")

print("✅ Dataset loaded. Number of documents:", len(df))

# =========================
# 3. BUILD RETRIEVER
# =========================

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["text"])

print("✅ TF-IDF vectorizer ready.")

# =========================
# 4. RETRIEVAL FUNCTION
# =========================


def retrieve_documents(query, top_k=3):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = df.iloc[top_indices].copy()
    results["score"] = similarities[top_indices]

    return results


# =========================
# 5. LLM SETUP (OPENROUTER)
# =========================

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)

# =========================
# 6. GENERATION FUNCTION
# =========================


def generate_answer(query, retrieved_docs):
    context = "\n".join(retrieved_docs["text"].tolist())

    prompt = f"""
You are a helpful medical assistant.

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}
"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
        )

        return response.choices[0].message.content

    except Exception as e:
        print("⚠️ LLM error:", e)
        print("Returning fallback answer...\n")

        # fallback: return retrieved docs
        return "Fallback answer:\n" + context


# =========================
# 7. MAIN LOOP
# =========================

print("\n💊 Drug Knowledge Assistant is ready!")
print("Type 'exit' to quit.\n")

while True:
    query = input("🔎 Ask a question: ")

    if query.lower() == "exit":
        print("👋 Goodbye!")
        break

    # Step 1: Retrieve
    retrieved = retrieve_documents(query)

    print("\n📄 Retrieved Documents:")
    for _i, row in retrieved.iterrows():
        print(f"- {row['text']} (score={row['score']:.3f})")

    # Step 2: Generate
    answer = generate_answer(query, retrieved)

    print("\n🤖 Answer:")
    print(answer)
    print("\n" + "=" * 50 + "\n")

✅ Dataset loaded. Number of documents: 100
✅ TF-IDF vectorizer ready.

💊 Drug Knowledge Assistant is ready!
Type 'exit' to quit.


📄 Retrieved Documents:
- Drug F is used to treat asthma. It helps open airways and improve breathing. (score=0.413)
- Side effects of Drug I include headache nausea and fatigue. (score=0.188)
- Side effects of Drug Y include nausea dizziness and hallucinations. (score=0.160)

🤖 Answer:
Drug F is used for asthma. I don't know its side effects.


👋 Goodbye!
